In [1]:
import scanpy as sc
import pandas as pd
import numpy as np

In [2]:
# Guide assign gene expression matrix
gex_a = sc.read_h5ad('/Users/chandrima.modak/Gladstone Dropbox/Chandrima Modak/gw-CRISPRa_from_cluster/D1_392i_Rest_lane1/D1_392i_Rest_gex_guide.h5ad')
gex_a.obs.head(n=5)

,library_id,lane_id,n_genes_by_counts,log1p_n_genes_by_counts,total_counts,log1p_total_counts,pct_counts_in_top_50_genes,pct_counts_in_top_100_genes,pct_counts_in_top_200_genes,pct_counts_in_top_500_genes,total_counts_mt,log1p_total_counts_mt,pct_counts_mt,n_genes,gRNA,guide_id,target_gene,UMI_counts
AAACAAGCAAGTTTCCACAGACCT-1_lane1_D1_392i_Rest,D1_392i_Rest,lane1,1266,7.144407,1755.0,7.470794,17.321937,25.071225,36.467236,56.353276,0.0,0.000000,0.000000,1266,ZC3H12A_A_CRISPRi_2,ZC3H12A_A_CRISPRi_2,ZC3H12A,3.0
AAACAAGCACAACAATACAGACCT-1_lane1_D1_392i_Rest,D1_392i_Rest,lane1,2263,7.724888,3902.0,8.269501,18.682727,24.243977,32.829318,50.076884,17.0,2.890372,0.435674,2263,VAMP8_-_85788707.23-P2_CRISPRi,VAMP8_-_85788707.23-P2_CRISPRi,VAMP8,77.0
AAACAAGCACCGGCAAACAGACCT-1_lane1_D1_392i_Rest,D1_392i_Rest,lane1,2211,7.701652,3916.0,8.273082,20.939734,27.093973,35.750766,52.936670,23.0,3.178054,0.587334,2211,NTC-580,NTC-580,NTC,89.0
AAACAAGCACGTAGGTACAGACCT-1_lane1_D1_392i_Rest,D1_392i_Rest,lane1,1722,7.451822,2666.0,7.888710,17.291823,23.555889,33.383346,54.163541,16.0,2.833213,0.600150,1722,MAPK3_+_30134158.23-P1P2_CRISPRi,MAPK3_+_30134158.23-P1P2_CRISPRi,MAPK3,113.0
AAACAAGCACTAACTCACAGACCT-1_lane1_D1_392i_Rest,D1_392i_Rest,lane1,2149,7.673223,3607.0,8.190908,17.687829,23.703909,32.492376,50.346548,21.0,3.091043,0.582201,2149,NTC-460,NTC-460,NTC,222.0


In [3]:
crispr_a = sc.read_h5ad('/Users/chandrima.modak/Gladstone Dropbox/Chandrima Modak/gw-CRISPRa_from_cluster/D1_392i_Rest_lane1/D1_392i_Rest_crispr_preprocessed.h5ad')
crispr_a.obs.head(n=5)

,library_id,lane_id,n_genes_by_counts,log1p_n_genes_by_counts,total_counts,log1p_total_counts,pct_counts_in_top_50_genes,pct_counts_in_top_100_genes,pct_counts_in_top_200_genes,pct_counts_in_top_500_genes
AAACAAGCAAGTTTCCACAGACCT-1_lane1_D1_392i_Rest,D1_392i_Rest,lane1,4,1.609438,6.0,1.945910,100.0,100.0,100.0,100.0
AAACAAGCACAACAATACAGACCT-1_lane1_D1_392i_Rest,D1_392i_Rest,lane1,2,1.098612,78.0,4.369448,100.0,100.0,100.0,100.0
AAACAAGCACCGGCAAACAGACCT-1_lane1_D1_392i_Rest,D1_392i_Rest,lane1,1,0.693147,89.0,4.499810,100.0,100.0,100.0,100.0
AAACAAGCACGTAGGTACAGACCT-1_lane1_D1_392i_Rest,D1_392i_Rest,lane1,3,1.386294,115.0,4.753590,100.0,100.0,100.0,100.0
AAACAAGCACTAACTCACAGACCT-1_lane1_D1_392i_Rest,D1_392i_Rest,lane1,4,1.609438,225.0,5.420535,100.0,100.0,100.0,100.0


In [4]:
# Sanity check on cells
diff = crispr_a.obs.index.difference(gex_a.obs.index)
len(diff)

11

In [5]:
# Checking the guide assignment in cell by type and count
guide_assigned_df = pd.read_csv('/Users/chandrima.modak/Gladstone Dropbox/Chandrima Modak/gw-CRISPRa_from_cluster/D1_392i_Rest_lane1/guide_assigned.csv')
guide_assigned_df_merge = guide_assigned_df.groupby('cell').agg(
    { 'cell':'size',
      'gRNA': list
    }
)
guide_assigned_df_merge['num_guide_assigned'] = guide_assigned_df_merge['gRNA'].apply(lambda x: len(x))
del guide_assigned_df_merge['cell']

In [7]:
guide_assigned_df_merge.shape

(23277, 2)

In [8]:
gex_df = gex_a.obs.copy()
gex_df.dropna(inplace = True)
gex_df.shape

(23274, 18)

In [9]:
# Sanity check on cells match the gene expression object
identical = guide_assigned_df_merge.index.intersection(gex_df.index)
diff = guide_assigned_df_merge.index.difference(identical)
len(diff)

3

In [20]:
def listing_target_gene(x):
    gene_list = []
    for i in x:
        i = i.split('_')[0]
        gene_list.append(i)
    return gene_list

In [22]:
guide_assigned_df_merge['target_gene'] = guide_assigned_df_merge['gRNA'].apply(lambda x : listing_target_gene(x))
guide_assigned_df_merge

,gRNA,num_guide_assigned,target_gene
cell,,,
AAACAAGCAAGTTTCCACAGACCT-1_lane1_D1_392i_Rest,[ZC3H12A_A_CRISPRi_2],1,[ZC3H12A]
AAACAAGCACAACAATACAGACCT-1_lane1_D1_392i_Rest,[VAMP8_-_85788707.23-P2_CRISPRi],1,[VAMP8]
AAACAAGCACCGGCAAACAGACCT-1_lane1_D1_392i_Rest,[NTC-580],1,[NTC-580]
AAACAAGCACGTAGGTACAGACCT-1_lane1_D1_392i_Rest,[MAPK3_+_30134158.23-P1P2_CRISPRi],1,[MAPK3]
AAACAAGCACTAACTCACAGACCT-1_lane1_D1_392i_Rest,[NTC-460],1,[NTC-460]
...,...,...,...
TTTGTGAGTGCTTCTCACAGACCT-1_lane1_D1_392i_Rest,"[MAPK3_A_CRISPRi_2, MAPK3_A_CRISPRi_3, AP5M1_+...",3,"[MAPK3, MAPK3, AP5M1]"
TTTGTGAGTGTGACCCACAGACCT-1_lane1_D1_392i_Rest,"[BATF_A_CRISPRi_3, PHF14_A_CRISPRi_2]",2,"[BATF, PHF14]"
TTTGTGAGTTAAGGCAACAGACCT-1_lane1_D1_392i_Rest,[ID3_A_CRISPRi_3],1,[ID3]


In [ ]:
# Sanity checks for guides assigned 
crispr_var = crispr_a.var.copy()
crispr_var.index

In [ ]:
guide_list = crispr_var.index.difference(guide_assigned_df.gRNA.unique())
len(guide_list)

In [ ]:
guide_threshold_df = pd.read_csv('/Users/chandrima.modak/Gladstone Dropbox/Chandrima Modak/gw-CRISPRa_from_cluster/D1_392i_Rest_lane1/guide_threshold.csv')
guide_threshold_df.set_index('gRNA', inplace=True)
guide_threshold_df.shape[0]

In [ ]:
crispr_var.shape[0] - guide_threshold_df.shape[0]